# LLM Evaluation (With and Without RAG)

This notebook evaluates the performance of **Llama 3** (local) and **Gemini API** on our synthetic dataset.
We evaluate them in two conditions:
1. **Baseline (No RAG):** The LLM answers the question using only its pre-trained knowledge.
2. **RAG:** The LLM answers the question with the context retrieved from our database.

We use the `ragas` framework to compute `Faithfulness` and `Answer Relevance`.

In [ ]:
import os
import json
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

# Langchain Wrappers for RAG
# Note: Ensure you have your retriever and chains implemented in src/
# from src.chains import setup_rag_chain, setup_baseline_chain

EVAL_FILE = "../data/evaluation_dataset.json"

## 1. Load Evaluation Dataset

In [ ]:
with open(EVAL_FILE, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)
    
df_eval = pd.DataFrame(eval_data)
print(f"Loaded {len(df_eval)} evaluation questions.")
df_eval.head(3)

## 2. Generate Answers (Mock Implementation)
In a real run, this loop will call your `src` code to generate the answers.

In [ ]:
def generate_all_answers(questions):
    results = {
        "question": [],
        "ground_truth": [],
        "contexts": [],
        "answer_gemini_baseline": [],
        "answer_gemini_rag": [],
        "answer_llama_baseline": [],
        "answer_llama_rag": []
    }
    
    # MOCK LOOP - Replace with actual chain invocation
    for item in questions:
        q = item["question"]
        gt = item["ground_truth"]
        ctx = [item["context"]]
        
        results["question"].append(q)
        results["ground_truth"].append(gt)
        results["contexts"].append(ctx) # Ragas expects a list of contexts
        
        # Execute baseline models (No context passed)
        # ans_gem_base = baseline_gemini_chain.invoke(q)
        results["answer_gemini_baseline"].append("[Gemini Base Answer]")
        results["answer_llama_baseline"].append("[Llama Base Answer]")
        
        # Execute RAG models (Retrieves context from FAISS)
        # ans_gem_rag = rag_gemini_chain.invoke(q)
        results["answer_gemini_rag"].append("[Gemini RAG Answer]")
        results["answer_llama_rag"].append("[Llama RAG Answer]")
        
    return results

# result_dict = generate_all_answers(eval_data)
# df_results = pd.DataFrame(result_dict)
# df_results.to_csv("../data/evaluation_results.csv", index=False)

## 3. Evaluate using RAGAS
We convert the pandas dataframe to a HuggingFace Dataset, then use Ragas to compute metrics.

In [ ]:
# Format required by Ragas: question, answer, contexts, ground_truth
def evaluate_model(df, answer_column_name):
    data_for_ragas = {
        "question": df["question"].tolist(),
        "answer": df[answer_column_name].tolist(),
        "contexts": df["contexts"].tolist(),
        "ground_truth": df["ground_truth"].tolist()
    }
    dataset = Dataset.from_dict(data_for_ragas)
    
    # Ragas uses OpenAI by default. Can override with Langchain LLMs if needed.
    result = evaluate(
        dataset,
        metrics=[
            faithfulness,
            answer_relevancy
        ]
    )
    return result

# print("Evaluating Gemini RAG...")
# gemini_rag_scores = evaluate_model(df_results, "answer_gemini_rag")
# print(gemini_rag_scores)